In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import pandas as pd

## IMPORTANTE

Para no ejecutar el código mil veces, he ido guardando los archivos en csv, estos pesan mas de 100mb por tanto no los puedo subir al github. Es decir que como mínimo se deberá ejecutar el código 1 vez para generar estos archivos.

Si ya se han generado, hay que saltar las ejecuciones donde se generen, estas son las que llamen a la función "factorizacion_ponderada_SGD_con_mascara"

In [2]:
df = pd.read_csv('BBDD_1M/ratings.dat', delimiter='::', engine='python', header=None, names=['userId', 'movieId', 'rating', 'timestamp'], encoding='ISO-8859-1')
indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
matriz_usuario_pelicula=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')

def normalizar_datos(matriz_escasez):
    # Creamos una copia de la matriz para evitar modificar el original
    matriz_escasez_copy = matriz_escasez.copy()
    
    # Inicializamos StandardScaler sin centrado en 0 debido a NaNs
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Aplicamos la normalización solo en las columnas que tienen datos no NaN
    for user_id in matriz_escasez_copy.index:
        # Seleccionamos las calificaciones del usuario (excluyendo NaNs)
        user_ratings = matriz_escasez_copy.loc[user_id].dropna()
        if not user_ratings.empty:
            # Normalizamos las calificaciones de este usuario
            normalized_ratings = scaler.fit_transform(user_ratings.values.reshape(-1, 1)).flatten()
            # Colocamos los valores normalizados en la matriz original, manteniendo NaNs donde no hay calificaciones
            matriz_escasez_copy.loc[user_id, user_ratings.index] = normalized_ratings
    
    # Llenamos los NaNs con 0
    matriz_escasez_copy = matriz_escasez_copy.fillna(0)
    return matriz_escasez_copy

matriz_normalizada = normalizar_datos(matriz_usuario_pelicula)
matriz_normalizada.head(20)


movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,1.202827,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.0,0.000000,0.0,-1.014718,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
6,0.119523,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
7,0.000000,0.000000,0.0,0.000000,0.0,-0.438529,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
8,0.124848,0.000000,0.0,-0.959767,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
9,1.548952,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,-0.901629,0.0,0.0,0.0,0.0


## Factorización ponderada de matrices

In [3]:
# Función para inicializar los factores de usuario y película
def inicializar_factores(num_usuarios, num_items, num_factors):
    # Inicializa las matrices U y V con valores aleatorios pequeños
    U = np.random.normal(scale=0.01, size=(num_usuarios, num_factors))
    V = np.random.normal(scale=0.01, size=(num_items, num_factors))
    return U, V

# Función para aplicar WMF
def factorizacion_ponderada_SGD_con_mascara(matriz, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv):
    num_usuarios, num_items = matriz.shape
    U, V = inicializar_factores(num_usuarios, num_items, num_factors)

    # Crear una máscara donde las entradas existentes tienen peso 1, las faltantes 0
    mascara = (matriz != 0).astype(float)

    for iteracion in range(num_iteraciones):
        for i in range(num_usuarios):
            for j in range(num_items):
                # Actualizar solo las entradas observadas
                if mascara[i, j] == 1:
                    error = matriz[i, j] - np.dot(U[i, :], V[j, :])
                    U[i, :] += learning_rate * (error * V[j, :] - regularizacion * U[i, :])
                    V[j, :] += learning_rate * (error * U[i, :] - regularizacion * V[j, :])

        # Calcular el error cuadrático medio solo para las entradas observadas
        mse = np.mean((mascara * (matriz - (U @ V.T))) ** 2)
        print(f"Iteración {iteracion + 1}/{num_iteraciones}, MSE: {mse:.4f}")

    # Generar las predicciones completas
    predicciones_completas = np.dot(U, V.T)

    # Convertir las predicciones a un DataFrame con índices y columnas originales
    predicciones_df = pd.DataFrame(predicciones_completas, index=matriz_normalizada.index, columns=matriz_normalizada.columns)

    # Guardar el DataFrame como un archivo CSV
    predicciones_df.to_csv(output_csv, index=True)
    print(f"Predicciones guardadas en {output_csv}")

    return U, V, predicciones_df

Vamos a predecir los valores faltantes

In [4]:
# Parámetros
output_csv = "FPM_1M/1M_usuario_pelicula_datos_simulados.csv"
num_factors = 10          # Número de factores latentes
num_iteraciones = 50      # Número de iteraciones
learning_rate = 0.05      # Tasa de aprendizaje
regularizacion = 0.1      # Parámetro de regularización

# Convertimos la matriz normalizada a numpy array
matriz_numpy = matriz_normalizada.values

# Aplicamos la factorización ponderada
U, V, predicciones_simuladas_df = factorizacion_ponderada_SGD_con_mascara(
    matriz_numpy, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv
)

# Predicciones completas
predicciones_completas = np.dot(U, V.T)

# Crear una máscara para identificar las entradas faltantes
mascara = (matriz_numpy != 0).astype(float)

# Predicciones solo para las entradas faltantes
predicciones_simuladas = (1 - mascara) * predicciones_completas

# Convertimos a DataFrame para visualizar mejor
predicciones_simuladas_df = pd.DataFrame(
    predicciones_simuladas, index=matriz_normalizada.index, columns=matriz_normalizada.columns
)

predicciones_simuladas_df.head()

Iteración 1/50, MSE: 0.0381
Iteración 2/50, MSE: 0.0351
Iteración 3/50, MSE: 0.0340
Iteración 4/50, MSE: 0.0335
Iteración 5/50, MSE: 0.0332
Iteración 6/50, MSE: 0.0329
Iteración 7/50, MSE: 0.0326
Iteración 8/50, MSE: 0.0323
Iteración 9/50, MSE: 0.0320
Iteración 10/50, MSE: 0.0318
Iteración 11/50, MSE: 0.0316
Iteración 12/50, MSE: 0.0315
Iteración 13/50, MSE: 0.0314
Iteración 14/50, MSE: 0.0313
Iteración 15/50, MSE: 0.0312
Iteración 16/50, MSE: 0.0311
Iteración 17/50, MSE: 0.0310
Iteración 18/50, MSE: 0.0310
Iteración 19/50, MSE: 0.0309
Iteración 20/50, MSE: 0.0309
Iteración 21/50, MSE: 0.0309
Iteración 22/50, MSE: 0.0308
Iteración 23/50, MSE: 0.0308
Iteración 24/50, MSE: 0.0308
Iteración 25/50, MSE: 0.0307
Iteración 26/50, MSE: 0.0307
Iteración 27/50, MSE: 0.0307
Iteración 28/50, MSE: 0.0307
Iteración 29/50, MSE: 0.0307
Iteración 30/50, MSE: 0.0307
Iteración 31/50, MSE: 0.0307
Iteración 32/50, MSE: 0.0307
Iteración 33/50, MSE: 0.0307
Iteración 34/50, MSE: 0.0306
Iteración 35/50, MSE: 0

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.000000,-0.233374,-0.062902,-0.272436,-0.064980,0.072753,-0.085564,0.240520,-0.050056,-0.126665,...,-0.859630,-0.021858,-0.411383,-0.696866,0.043101,-0.278260,-0.303478,0.124165,0.110917,-0.170537
2,0.349293,-0.391535,-0.403302,-0.457889,-0.283650,0.148962,0.015075,-0.325754,-0.493883,-0.231079,...,-0.694385,-0.272663,-1.282194,-1.428121,-0.803192,-0.250323,-0.284557,0.082375,0.096837,0.008688
3,-0.000340,-0.250678,-0.076430,-0.336880,-0.205802,0.249806,0.144345,-0.076517,-0.430832,-0.056983,...,-0.870787,-0.023655,-0.654958,-0.887197,-0.226332,-0.256688,-0.227378,0.183240,0.127213,-0.094462
4,0.141634,-0.572816,-0.935219,-0.650345,-0.698975,0.296924,-0.460581,-0.524865,-0.413829,-0.518186,...,-0.996556,-0.287715,-1.224937,-1.374275,-0.096522,-0.265095,0.163068,-0.067443,0.669038,-0.058169
5,0.223276,-0.370559,-0.676071,-0.467846,-0.769167,0.000000,-0.594125,-0.409511,-0.291147,-0.341720,...,0.342739,-0.362672,-0.740002,-0.604052,0.052076,-0.310195,0.502066,-0.086639,0.270250,-0.175998


Juntamos ambas matrices

In [5]:
matriz_simulada = pd.read_csv("FPM_1M/1M_usuario_pelicula_datos_simulados.csv", index_col = 0)
# Aseguramos que los índices y columnas coincidan
matriz_simulada.columns = matriz_normalizada.columns
matriz_simulada.index = matriz_normalizada.index

matriz_completa = matriz_normalizada.copy()
matriz_completa = matriz_completa.where(matriz_completa != 0, matriz_simulada)

matriz_completa.head(6)

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,1.202827,-0.233374,-0.062902,-0.272436,-0.064980,0.072753,-0.085564,0.240520,-0.050056,-0.126665,...,-0.859630,-0.021858,-0.411383,-0.696866,0.043101,-0.278260,-0.303478,0.124165,0.110917,-0.170537
2,0.349293,-0.391535,-0.403302,-0.457889,-0.283650,0.148962,0.015075,-0.325754,-0.493883,-0.231079,...,-0.694385,-0.272663,-1.282194,-1.428121,-0.803192,-0.250323,-0.284557,0.082375,0.096837,0.008688
3,-0.000340,-0.250678,-0.076430,-0.336880,-0.205802,0.249806,0.144345,-0.076517,-0.430832,-0.056983,...,-0.870787,-0.023655,-0.654958,-0.887197,-0.226332,-0.256688,-0.227378,0.183240,0.127213,-0.094462
4,0.141634,-0.572816,-0.935219,-0.650345,-0.698975,0.296924,-0.460581,-0.524865,-0.413829,-0.518186,...,-0.996556,-0.287715,-1.224937,-1.374275,-0.096522,-0.265095,0.163068,-0.067443,0.669038,-0.058169
5,0.223276,-0.370559,-0.676071,-0.467846,-0.769167,-1.014718,-0.594125,-0.409511,-0.291147,-0.341720,...,0.342739,-0.362672,-0.740002,-0.604052,0.052076,-0.310195,0.502066,-0.086639,0.270250,-0.175998
6,0.119523,-0.003177,-0.065147,0.008457,0.238358,-0.179790,0.245088,-0.058977,0.044439,0.032477,...,-0.546814,-0.085168,-0.251715,-0.376364,-0.353465,-0.058314,-0.628791,-0.017153,-0.124341,0.131441


### Vamos a reescalar los valores, tanto los simulados como los normalizados sobre los valores originales

Primero los normalizados

In [4]:
def reescalar_matriz_normalizada(predicciones_normalizadas, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa sólo donde había datos originales
    predicciones_reescaladas = predicciones_normalizadas.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    predicciones_reescaladas = predicciones_reescaladas.where(~matriz_usuario_pelicula.isna(), np.nan)
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas

In [7]:
matriz_normalizada_reescalada = reescalar_matriz_normalizada(matriz_normalizada, matriz_usuario_pelicula)
matriz_normalizada_reescalada.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN


Luego los simulados

In [5]:
def reescalar_matriz_simulada(matriz_simulada, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa a toda la matriz simulada
    predicciones_reescaladas = matriz_simulada.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    
    # Redondeamos y limitamos los valores dentro del rango permitido
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas


In [6]:
def reescalar_solo_simulados(matriz_simulada, matriz_usuario_pelicula_original, min_rating=0.5, max_rating=5.0):
    """
    Reescala solo los valores simulados en la matriz simulada, utilizando las estadísticas de la matriz original.
    """
    # Crear una máscara de los valores simulados (donde matriz_usuario_pelicula_original tiene NaN)
    mascara_simulados = matriz_usuario_pelicula_original.isna()

    # Verificar alineación
    if not matriz_simulada.index.equals(matriz_usuario_pelicula_original.index) or not matriz_simulada.columns.equals(matriz_usuario_pelicula_original.columns):
        raise ValueError("Índices o columnas de las matrices no están alineados.")

    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)

    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)

    # Crear una copia para trabajar únicamente con los valores simulados
    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulados] = np.nan  # Mantener solo los valores simulados

    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)

    return valores_simulados


In [11]:
# Reescalar solo los valores simulados
valores_simulados_reescalados = reescalar_solo_simulados(
    matriz_simulada=matriz_simulada,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    min_rating=0.5,
    max_rating=5.0
)

valores_simulados_reescalados_a_csv = valores_simulados_reescalados.fillna(0)
valores_simulados_reescalados_a_csv.to_csv("FPM_1M/1M_usuario_pelicula_datos_simulados_reescalados.csv", index=True)

# Mostrar las primeras filas de la matriz reescalada con solo valores simulados
print("Valores Simulados Reescalados (solo simulados, 2 decimales):")
valores_simulados_reescalados.head(6)


Valores Simulados Reescalados (solo simulados, 2 decimales):


movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,4.03,4.15,4.00,4.14,4.24,4.13,4.35,4.15,4.10,...,3.60,4.17,3.91,3.71,4.22,4.00,3.98,4.27,4.26,4.07
2,4.06,3.32,3.31,3.25,3.43,3.86,3.73,3.39,3.22,3.48,...,3.02,3.44,2.43,2.28,2.91,3.46,3.43,3.80,3.81,3.72
3,3.90,3.66,3.83,3.57,3.70,4.15,4.04,3.83,3.48,3.85,...,3.04,3.88,3.26,3.03,3.68,3.65,3.68,4.08,4.03,3.81
4,4.34,3.57,3.18,3.49,3.44,4.51,3.69,3.62,3.74,3.63,...,3.12,3.88,2.87,2.71,4.09,3.90,4.37,4.12,4.91,4.13
5,3.40,2.73,2.38,2.62,2.28,NaN,2.47,2.68,2.82,2.76,...,3.53,2.74,2.31,2.46,3.21,2.80,3.72,3.05,3.45,2.95
6,NaN,3.90,3.85,3.91,4.10,3.75,4.11,3.85,3.94,3.93,...,3.45,3.83,3.69,3.59,3.61,3.85,3.38,3.89,3.80,4.01


Matriz completa

In [12]:
# Aseguramos que las matrices tienen índices y columnas alineados
valores_simulados_reescalados.columns = matriz_normalizada.columns
valores_simulados_reescalados.index = matriz_normalizada.index

# Crear la matriz completa reescalada
matriz_completa_reescalada = matriz_normalizada_reescalada.copy()
matriz_completa_reescalada = matriz_completa_reescalada.where(~pd.isna(matriz_completa_reescalada), valores_simulados_reescalados)

matriz_completa_reescalada.head(6)


movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,5.00,4.03,4.15,4.00,4.14,4.24,4.13,4.35,4.15,4.10,...,3.60,4.17,3.91,3.71,4.22,4.00,3.98,4.27,4.26,4.07
2,4.06,3.32,3.31,3.25,3.43,3.86,3.73,3.39,3.22,3.48,...,3.02,3.44,2.43,2.28,2.91,3.46,3.43,3.80,3.81,3.72
3,3.90,3.66,3.83,3.57,3.70,4.15,4.04,3.83,3.48,3.85,...,3.04,3.88,3.26,3.03,3.68,3.65,3.68,4.08,4.03,3.81
4,4.34,3.57,3.18,3.49,3.44,4.51,3.69,3.62,3.74,3.63,...,3.12,3.88,2.87,2.71,4.09,3.90,4.37,4.12,4.91,4.13
5,3.40,2.73,2.38,2.62,2.28,2.00,2.47,2.68,2.82,2.76,...,3.53,2.74,2.31,2.46,3.21,2.80,3.72,3.05,3.45,2.95
6,4.00,3.90,3.85,3.91,4.10,3.75,4.11,3.85,3.94,3.93,...,3.45,3.83,3.69,3.59,3.61,3.85,3.38,3.89,3.80,4.01


Como en el ejemplo anterior con KNN voy a ver si la media de simulacion de los datos simulados para el usuario 3598 que tenía originalmente una media de 1.01 se acercan o no

In [16]:
valores_simulados_usuario_442 = valores_simulados_reescalados.loc[3598]
puntuaciones_simuladas = valores_simulados_usuario_442.dropna()
media_simulada = puntuaciones_simuladas.mean()

print(f"Media de puntuación simulada del usuario 3598: {media_simulada:.2f}")


Media de puntuación simulada del usuario 3598: 1.02


Los datos son muy similares, ahora tocaria mirar el accuracy prediciendo algunos valores originales aleatorios. En concreto voy a seleccionar un 10% aleatorio sobre la máscara de la matriz original para simular datos, ese 10% serán valores originales que voy a simular para posteriormente comparar, si los resultados son satisfactorios, la factorización ponderada de matrices desarrollada será válida

## Accuracy

Primero seleccionamos un 10% aleatorio de datos originales

In [7]:
def generar_factorizacion(matriz_original, mascara_simulacion, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv="FPM_1M/evaluacion_10%.csv"):
    """
    Simula exclusivamente los valores seleccionados por la máscara.
    """
    # Crear una copia de la matriz y aplicar la máscara de simulación
    matriz_modificada = matriz_original.copy()
    matriz_modificada[mascara_simulacion] = 0  # Eliminar temporalmente los valores seleccionados para simulación
    matriz_modificada = matriz_modificada.fillna(0)  # Reemplazar NaN por 0 para evitar errores

    # Aplicar la factorización ponderada
    U, V,fpm = factorizacion_ponderada_SGD_con_mascara(matriz_modificada.values, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv)

    predicciones_completas = np.dot(U, V.T)
    predicciones_completas_df = pd.DataFrame(predicciones_completas, index=matriz_original.index, columns=matriz_original.columns)
    
    return predicciones_completas_df

In [8]:
def recortar_por_mascara(predicciones_completas, mascara_simulacion):
    """
    Recorta las predicciones completas utilizando la máscara de simulación.
    Devuelve un DataFrame con NaN en las posiciones fuera de la máscara.

    Args:
    - predicciones_completas (DataFrame): Predicciones generadas para toda la matriz.
    - mascara_simulacion (DataFrame): Máscara booleana que indica las posiciones a conservar.

    Returns:
    - DataFrame: Predicciones recortadas según la máscara.
    """
    # Crear un DataFrame con NaN en todas las posiciones
    predicciones_recortadas = pd.DataFrame(
        np.nan, index=predicciones_completas.index, columns=predicciones_completas.columns
    )
    # Conservar solo las posiciones seleccionadas por la máscara
    predicciones_recortadas[mascara_simulacion] = predicciones_completas[mascara_simulacion]

    return predicciones_recortadas

In [9]:
def reescalar_simulados_con_mascara(matriz_simulada, matriz_usuario_pelicula_original, mascara_simulacion, min_rating=0.5, max_rating=5.0):
    """
    Reescala únicamente los valores simulados seleccionados por la máscara, utilizando las estadísticas de la matriz original.
    """
    # Verificar alineación
    if not matriz_simulada.index.equals(matriz_usuario_pelicula_original.index) or \
       not matriz_simulada.columns.equals(matriz_usuario_pelicula_original.columns):
        raise ValueError("Índices o columnas de las matrices no están alineados.")
    if not matriz_simulada.index.equals(mascara_simulacion.index) or \
       not matriz_simulada.columns.equals(mascara_simulacion.columns):
        raise ValueError("Índices o columnas de la máscara no están alineados con la matriz simulada.")

    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)

    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)

    # Crear una copia para trabajar únicamente con los valores simulados según la máscara
    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulacion] = np.nan  # Mantener solo los valores seleccionados por la máscara

    # Aplicar el reescalado a los valores seleccionados
    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)

    return valores_simulados


In [10]:
num_factors = 10
num_iteraciones = 50
learning_rate = 0.05
regularizacion = 0.1

mascara_original = ~matriz_usuario_pelicula.isna()
num_datos = mascara_original.sum().sum()
num_datos_a_simular = int(0.1 * num_datos)

indices_aleatorios = np.random.choice(
    mascara_original.stack()[mascara_original.stack()].index,
    size=num_datos_a_simular,
    replace=False
)

mascara_simulacion = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
for fila, columna in indices_aleatorios:
    mascara_simulacion.loc[fila, columna] = True

Vamos a mirar si los valores pertenecen a la máscara

In [22]:
num_valores_validos = mascara_original.sum().sum()
num_simulados = mascara_simulacion.sum().sum()
porcentaje_simulados = (num_simulados / num_valores_validos) * 100

print(f"Total de valores válidos en la matriz original: {num_valores_validos}")
print(f"Total de valores seleccionados para simulación: {num_simulados}")
print(f"Porcentaje de valores simulados: {porcentaje_simulados:.2f}%")

Total de valores válidos en la matriz original: 1000209
Total de valores seleccionados para simulación: 100020
Porcentaje de valores simulados: 10.00%


In [23]:
predicciones_completas = generar_factorizacion(
    matriz_original=matriz_normalizada,
    mascara_simulacion=mascara_simulacion,
    num_factors=num_factors,
    num_iteraciones=num_iteraciones,
    learning_rate=learning_rate,
    regularizacion=regularizacion
)

Iteración 1/50, MSE: 0.0345
Iteración 2/50, MSE: 0.0318
Iteración 3/50, MSE: 0.0307
Iteración 4/50, MSE: 0.0301
Iteración 5/50, MSE: 0.0298
Iteración 6/50, MSE: 0.0295
Iteración 7/50, MSE: 0.0293
Iteración 8/50, MSE: 0.0290
Iteración 9/50, MSE: 0.0287
Iteración 10/50, MSE: 0.0285
Iteración 11/50, MSE: 0.0283
Iteración 12/50, MSE: 0.0282
Iteración 13/50, MSE: 0.0281
Iteración 14/50, MSE: 0.0279
Iteración 15/50, MSE: 0.0278
Iteración 16/50, MSE: 0.0278
Iteración 17/50, MSE: 0.0277
Iteración 18/50, MSE: 0.0276
Iteración 19/50, MSE: 0.0276
Iteración 20/50, MSE: 0.0275
Iteración 21/50, MSE: 0.0275
Iteración 22/50, MSE: 0.0275
Iteración 23/50, MSE: 0.0274
Iteración 24/50, MSE: 0.0274
Iteración 25/50, MSE: 0.0274
Iteración 26/50, MSE: 0.0274
Iteración 27/50, MSE: 0.0273
Iteración 28/50, MSE: 0.0273
Iteración 29/50, MSE: 0.0273
Iteración 30/50, MSE: 0.0273
Iteración 31/50, MSE: 0.0273
Iteración 32/50, MSE: 0.0273
Iteración 33/50, MSE: 0.0273
Iteración 34/50, MSE: 0.0273
Iteración 35/50, MSE: 0

In [14]:
predicciones_10_por_ciento = pd.read_csv("FPM_1M/evaluacion_10%.csv",index_col=0)
predicciones_10_por_ciento.columns = predicciones_10_por_ciento.columns.astype(int)

predicciones_recortadas = recortar_por_mascara(predicciones_10_por_ciento, mascara_simulacion)
predicciones_recortadas.head()

,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
valores_reescalados_con_mascara = reescalar_simulados_con_mascara(
    matriz_simulada=predicciones_recortadas,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    mascara_simulacion=mascara_simulacion
)

print("Valores reescalados con máscara:")
valores_reescalados_con_mascara.head()

Valores reescalados con máscara:


,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
def calcular_mape_rmse_solo_simulados(valores_originales, valores_simulados):
    """
    Calcula el MAPE y el RMSE para los valores simulados en comparación con los valores originales.
    
    Args:
        valores_originales: DataFrame con las valoraciones originales.
        valores_simulados: DataFrame con las valoraciones simuladas.
    
    Returns:
        mape: Mean Absolute Percentage Error.
        rmse: Root Mean Square Error.
    """
    # Máscara de valores simulados válidos (no NaN)
    mascara_simulados = ~valores_simulados.isna()

    # Extraer los valores simulados y sus correspondientes originales
    valores_simulados_filtrados = valores_simulados[mascara_simulados]
    valores_originales_filtrados = valores_originales[mascara_simulados]

    # Calcular el MAPE
    errores_relativos = np.abs((valores_simulados_filtrados - valores_originales_filtrados) / valores_originales_filtrados)
    mape = errores_relativos.mean().mean() * 100  # Promedio de errores relativos en porcentaje

    # Calcular el RMSE
    errores_cuadraticos = (valores_simulados_filtrados - valores_originales_filtrados) ** 2
    rmse = np.sqrt(errores_cuadraticos.mean().mean())  # Promedio de errores cuadráticos

    return mape, rmse


In [17]:
mape, rmse = calcular_mape_rmse_solo_simulados(valores_originales=matriz_usuario_pelicula, valores_simulados=valores_reescalados_con_mascara)
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.4f}")

MAPE: 28.82%
RMSE: 0.8264


Puesto que estamos escogiendo un 10% de datos aleatorios cada ejecución mostrará datos aleatorios, para tener un resultado mas consistente vamos a ejecutar 10 veces para tener una media de MAPE y RMSE

In [23]:
def evaluar_factorizacion_por_semillas(
    matriz_usuario_pelicula, 
    predicciones_10_por_ciento, 
    num_iteraciones=10, 
    porcentaje_simulacion=0.1
):
    """
    Evalúa la factorización ponderada de matrices generando diferentes máscaras de simulación 
    usando semillas aleatorias y calcula el MAPE y RMSE promedio.
    
    Args:
        matriz_usuario_pelicula: DataFrame original de valoraciones de usuarios.
        predicciones_10_por_ciento: DataFrame con predicciones simuladas.
        num_iteraciones: Número de iteraciones (semillas) para ejecutar el proceso.
        porcentaje_simulacion: Porcentaje de datos a simular.

    Returns:
        resultados_df: DataFrame con los resultados de cada iteración (semilla).
        mape_promedio: MAPE promedio de las iteraciones.
        rmse_promedio: RMSE promedio de las iteraciones.
    """
    resultados = []
    
    for seed in range(num_iteraciones):
        np.random.seed(seed)

        # Generar máscara de simulación
        mascara_original = ~matriz_usuario_pelicula.isna()
        num_datos = mascara_original.sum().sum()
        num_datos_a_simular = int(porcentaje_simulacion * num_datos)
        indices_aleatorios = np.random.choice(
            mascara_original.stack()[mascara_original.stack()].index,
            size=num_datos_a_simular,
            replace=False
        )
        mascara_simulacion = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
        for fila, columna in indices_aleatorios:
            mascara_simulacion.loc[fila, columna] = True

        # Recortar predicciones por la máscara generada
        predicciones_recortadas = recortar_por_mascara(predicciones_10_por_ciento, mascara_simulacion)

        # Reescalar valores
        valores_reescalados_con_mascara = reescalar_simulados_con_mascara(
            matriz_simulada=predicciones_recortadas,
            matriz_usuario_pelicula_original=matriz_usuario_pelicula,
            mascara_simulacion=mascara_simulacion
        )

        # Calcular MAPE y RMSE
        mape, rmse = calcular_mape_rmse_solo_simulados(
            valores_originales=matriz_usuario_pelicula,
            valores_simulados=valores_reescalados_con_mascara
        )
        resultados.append({"Ejecución": seed + 1, "MAPE (%)": round(mape, 2), "RMSE": round(rmse, 4)})

    # Crear DataFrame con los resultados
    resultados_df = pd.DataFrame(resultados)

    # Calcular promedios
    mape_promedio = resultados_df["MAPE (%)"].mean()
    rmse_promedio = resultados_df["RMSE"].mean()

    return resultados_df, mape_promedio, rmse_promedio

In [24]:
predicciones_10_por_ciento = pd.read_csv("FPM_1M/evaluacion_10%.csv", index_col=0)
predicciones_10_por_ciento.columns = predicciones_10_por_ciento.columns.astype(int)

resultados_df, mape_promedio, rmse_promedio = evaluar_factorizacion_por_semillas(
    matriz_usuario_pelicula=matriz_usuario_pelicula,
    predicciones_10_por_ciento=predicciones_10_por_ciento,
    num_iteraciones=10,
    porcentaje_simulacion=0.1
)

In [27]:
print("\nResultados por ejecución:")
print(resultados_df.to_string(index=False))
print(f"\nMAPE promedio: {mape_promedio:.2f}%")
print(f"RMSE promedio: {rmse_promedio:.4f}")


Resultados por ejecución:
 Ejecución  MAPE (%)   RMSE
         1     28.80 0.8272
         2     28.51 0.8258
         3     28.71 0.8282
         4     28.73 0.8266
         5     28.70 0.8230
         6     28.87 0.8298
         7     28.83 0.8332
         8     28.72 0.8276
         9     28.82 0.8333
        10     29.16 0.8322

MAPE promedio: 28.79%
RMSE promedio: 0.8287
